# LD7187 — Fraud Detection Pipeline
**Run it in Google Colab (on GCP)**

This notebook runs the project in Colab, which is hosted on Google Cloud.

### Quick steps
1. Run the cells from top to bottom
2. Check outputs in `/content/outputs/`

## Cell 1 — Install packages

In [4]:
!pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost shap -q

## Cell 2 — Clone your GitHub repo

In [3]:
# Put your real GitHub repo URL here
GITHUB_REPO = "https://github.com/FahzainAhmad/fraud-detection-ml.git"

!git clone {GITHUB_REPO}

import os
# Move into the repo folder we just cloned
repo_name = GITHUB_REPO.split('/')[-1].replace('.git', '')
os.chdir(f'/content/{repo_name}')
print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

Cloning into 'fraud-detection-ml'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 34 (delta 8), reused 14 (delta 4), pack-reused 13 (from 1)
Receiving objects: 100% (34/34), 42.86 MiB | 14.44 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Working directory: /content/fraud-detection-ml
Files: ['main.py', 'creditcard.csv', 'colab_deploy.ipynb', '.git', 'eda.py', 'config.py', 'models.py', 'requirements.txt', 'evaluation.py', 'data_loader.py', 'preprocessing.py', 'README.md']


## Cell 3 — Run the full pipeline

In [5]:
import warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
warnings.filterwarnings('ignore')
np.random.seed(42)

from data_loader import load_data
from eda import run_eda
from preprocessing import preprocess
from models import train_all_models
from evaluation import evaluate_all

# Run the full pipeline
df = load_data()
run_eda(df)
X_train, y_train, X_test, y_test, feature_names = preprocess(df)
fraud_ratio = (df['Class'] == 0).sum() / (df['Class'] == 1).sum()
results = train_all_models(X_train, y_train, fraud_ratio)
metrics_df, thresh_df = evaluate_all(results, X_test, y_test)

print('\nPipeline complete!')


=== Loading Data ===
Shape          : (284807, 31)
Missing values : 0
Duplicates     : 1081

Class distribution:
Class
0    284315
1       492
Name: count, dtype: int64

Fraud rate     : 0.1727%
  → 492 fraudulent transactions out of 284,807 total

=== Task 2: Exploratory Data Analysis ===
  Saved -> outputs/01_class_distribution.png
  Saved -> outputs/02_amount_by_class.png
Amount statistics by class:
          count        mean         std  min   25%    50%     75%       max
Class                                                                      
0      284315.0   88.291022  250.105092  0.0  5.65  22.00   77.05  25691.16
1         492.0  122.211321  256.683288  0.0  1.00   9.25  105.89   2125.87
  Saved -> outputs/03_correlation_heatmap.png

Top features correlated with Class:
V17    0.326481
V14    0.302544
V12    0.260593
V10    0.216883
V16    0.196539
V3     0.192961
V7     0.187257
V11    0.154876
V4     0.133447
V18    0.111485
Name: Class, dtype: float64
  Saved -> outputs

## Cell 4 — Show output plots

In [6]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

output_files = sorted([
    f for f in os.listdir('outputs') if f.endswith('.png')
])

print(f"Total plots generated: {len(output_files)}")

for fname in output_files:
    img = mpimg.imread(f'outputs/{fname}')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(fname, fontsize=10)
    plt.tight_layout()
    plt.show()

Total plots generated: 24


## Cell 5 — Print final metrics

In [7]:
print('=== Final Model Metrics ===')
print(metrics_df.to_string(index=False))
print('\n=== Threshold Tuning ===')
print(thresh_df.to_string(index=False))

=== Final Model Metrics ===
              Model  Precision  Recall  F1 Score  ROC-AUC  PR-AUC
Logistic Regression     0.0541  0.8980    0.1021   0.9682  0.7152
      Random Forest     0.4565  0.8571    0.5957   0.9789  0.7983
            XGBoost     0.0529  0.9184    0.1000   0.9770  0.7200
     Neural Network     0.6412  0.8571    0.7336   0.9565  0.8637

=== Threshold Tuning ===
              Model  Default Threshold  Optimal Threshold  F1 @ Default  F1 @ Optimal
Logistic Regression                0.5             1.0000        0.1021        0.8211
      Random Forest                0.5             0.9103        0.5957        0.8111
            XGBoost                0.5             0.9995        0.1000        0.8085
     Neural Network                0.5             0.9980        0.7336        0.8667


## Cell 6 — Download all output plots

In [8]:
import shutil
from google.colab import files

# Zip the outputs folder, then download it
shutil.make_archive('/content/fraud_detection_outputs', 'zip', 'outputs')
files.download('/content/fraud_detection_outputs.zip')
print('Downloading outputs.zip ...')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>